# Day 6 — Advanced Analytics & Risk Metrics
**Bluestock Fintech Mutual Fund Analytics Capstone**
*Author: Kunal Gupta | Date: 2025-12-31*

This notebook covers six advanced analytical modules:
1. Historical VaR (95%) & CVaR
2. Rolling 90-Day Sharpe Ratio
3. Investor Cohort Analysis
4. SIP Continuity Analysis
5. Sector HHI Concentration
6. Advanced Insights Summary


## 0. Setup & Imports

In [1]:

import sys, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path

warnings.filterwarnings("ignore")

BASE_DIR          = Path().resolve().parent
DATA_RAW          = BASE_DIR / "data" / "raw"
DATA_SETS         = BASE_DIR / "data_sets"
DATA_PROCESSED    = BASE_DIR / "data" / "processed"
REPORTS_DIR       = BASE_DIR / "reports"

sys.path.insert(0, str(BASE_DIR / "scripts"))
sys.path.insert(0, str(BASE_DIR / "src"))

plt.rcParams.update({
    "figure.facecolor": "#0F172A",
    "axes.facecolor":   "#1E293B",
    "axes.edgecolor":   "#334155",
    "axes.labelcolor":  "#E2E8F0",
    "text.color":       "#E2E8F0",
    "xtick.color":      "#94A3B8",
    "ytick.color":      "#94A3B8",
    "grid.color":       "#334155",
    "grid.alpha":       0.4,
    "legend.facecolor": "#1E293B",
    "legend.edgecolor": "#334155",
    "font.family":      "DejaVu Sans",
})
EMERALD  = "#10B981"
BLUE     = "#3B82F6"
AMBER    = "#F59E0B"
RED      = "#EF4444"
SLATE    = "#94A3B8"
FUND_COLORS = [EMERALD, BLUE, AMBER, RED, "#A78BFA"]

# ── Load core datasets
df_nav  = pd.read_csv(DATA_PROCESSED / "clean_nav.csv", parse_dates=["date"])
df_nav.sort_values(["amfi_code", "date"], inplace=True)
df_nav["daily_return"] = df_nav.groupby("amfi_code")["nav"].pct_change()

df_tx   = pd.read_csv(DATA_PROCESSED / "clean_transactions.csv", parse_dates=["transaction_date"])
df_perf = pd.read_csv(DATA_SETS / "07_scheme_performance.csv")
df_port = pd.read_csv(DATA_SETS / "09_portfolio_holdings.csv")
df_fund = pd.read_csv(DATA_SETS / "01_fund_master.csv")

print(f"NAV records  : {len(df_nav):,}")
print(f"Transactions : {len(df_tx):,}")
print(f"Funds        : {df_nav['amfi_code'].nunique()}")


NAV records  : 51,400
Transactions : 32,778
Funds        : 40


---
## 1. Historical VaR (95%) & CVaR

**Value at Risk (VaR 95%)** — the worst daily loss that will NOT be exceeded on 95% of trading days
(i.e., the 5th percentile of daily return distribution).

**CVaR (Conditional VaR / Expected Shortfall)** — the average return on the worst 5% of days,
indicating expected loss in extreme market conditions.


In [2]:

RISK_FREE_ANNUAL = 0.065
TRADING_DAYS = 252

var_records = []
for amfi, grp in df_nav.groupby("amfi_code"):
    returns = grp["daily_return"].dropna()
    if len(returns) < 30:
        continue
    var_95  = np.percentile(returns, 5)              # 5th percentile
    cvar_95 = returns[returns <= var_95].mean()      # mean of returns below VaR
    ann_vol = returns.std() * np.sqrt(TRADING_DAYS) * 100
    var_records.append({
        "amfi_code"      : amfi,
        "var_95_pct"     : round(var_95 * 100, 4),
        "cvar_95_pct"    : round(cvar_95 * 100, 4),
        "ann_volatility" : round(ann_vol, 4),
        "n_observations" : len(returns),
    })

df_var = pd.DataFrame(var_records)

# Merge scheme names
name_map = df_perf[["amfi_code", "scheme_name", "fund_house"]].copy()
name_map["amfi_code"] = name_map["amfi_code"].astype(str)
df_var["amfi_code"] = df_var["amfi_code"].astype(str)
df_var = df_var.merge(name_map, on="amfi_code", how="left")

df_var.to_csv(DATA_PROCESSED / "var_cvar_report.csv", index=False)
print(f"VaR/CVaR computed for {len(df_var)} funds → var_cvar_report.csv")
df_var.sort_values("var_95_pct").head(10)[
    ["scheme_name", "var_95_pct", "cvar_95_pct", "ann_volatility"]
].to_string(index=False)


VaR/CVaR computed for 40 funds → var_cvar_report.csv


'                                   scheme_name  var_95_pct  cvar_95_pct  ann_volatility\n    SBI Small Cap Fund - Regular Plan - Growth    -11.5140     -19.0751        119.3265\n         DSP Small Cap Fund - Regular - Growth     -9.3315     -19.2843        115.2565\n     SBI Small Cap Fund - Direct Plan - Growth     -9.2594     -16.5692        110.1978\n Mirae Asset Tax Saver Fund - Regular - Growth     -8.1586     -14.7889         81.8138\n        Axis Small Cap Fund - Regular - Growth     -8.0276     -16.5607        109.4963\nNippon India Small Cap Fund - Regular - Growth     -7.9140     -14.2490         87.0421\n      ICICI Pru Midcap Fund - Regular - Growth     -7.6757     -17.6073        100.1421\n        ABSL Small Cap Fund - Regular - Growth     -7.6609     -14.1632         89.2775\n           Axis Midcap Fund - Regular - Growth     -6.9874     -16.3100         94.2384\n        Kotak Flexicap Fund - Regular - Growth     -6.9188     -16.0219         94.1621'

In [3]:

# ── Plot: Top 15 funds by VaR (most risky first) ──────────────────────────
top15_var = df_var.sort_values("var_95_pct").head(15).copy()
top15_var["short_name"] = top15_var["scheme_name"].str.split("-").str[0].str.strip()

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(top15_var["short_name"], top15_var["var_95_pct"], color=RED, alpha=0.85)
ax.barh(top15_var["short_name"], top15_var["cvar_95_pct"], color=AMBER, alpha=0.5,
        label="CVaR 95%")
ax.set_xlabel("Daily Loss % (negative = worse)")
ax.set_title("Historical VaR (95%) vs CVaR — Top 15 Highest Risk Funds", fontsize=14, pad=12)
ax.legend(["VaR 95%", "CVaR 95%"])
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig(REPORTS_DIR / "advanced_var_cvar.png", dpi=150, bbox_inches="tight",
            facecolor=fig.get_facecolor())
plt.show()
print("Chart saved → reports/advanced_var_cvar.png")


Chart saved → reports/advanced_var_cvar.png


---
## 2. Rolling 90-Day Sharpe Ratio

The rolling Sharpe shows how risk-adjusted returns evolved over market cycles
(2022 crash, 2023 bull run, 2024 correction, 2025 recovery).
We track 5 representative funds covering different categories.


In [4]:

KEY_FUNDS = {
    "119551" : "SBI Bluechip",
    "119598" : "SBI Small Cap",
    "149323" : "DSP Midcap",
    "118632" : "Nippon Large Cap",
    "119092" : "Axis Bluechip",
}
DAILY_RF = RISK_FREE_ANNUAL / TRADING_DAYS
WINDOW   = 90

fig, ax = plt.subplots(figsize=(14, 6))

for i, (amfi, label) in enumerate(KEY_FUNDS.items()):
    amfi_int = int(amfi)
    grp = df_nav[df_nav["amfi_code"] == amfi_int].set_index("date")["daily_return"].dropna()
    if len(grp) < WINDOW + 5:
        continue
    roll_mean  = grp.rolling(WINDOW).mean()
    roll_std   = grp.rolling(WINDOW).std()
    roll_sharpe = ((roll_mean - DAILY_RF) / roll_std) * np.sqrt(TRADING_DAYS)
    ax.plot(roll_sharpe.index, roll_sharpe.values,
            label=label, color=FUND_COLORS[i], linewidth=1.6, alpha=0.9)

ax.axhline(0, color=SLATE, linestyle="--", linewidth=0.8, alpha=0.6)
ax.axhline(1, color=EMERALD, linestyle=":", linewidth=0.8, alpha=0.5, label="Sharpe = 1 (good)")
ax.set_xlabel("Date")
ax.set_ylabel("Rolling 90-Day Sharpe Ratio")
ax.set_title("Rolling 90-Day Sharpe Ratio — 5 Key Funds (2022–2025)", fontsize=14, pad=12)
ax.legend(loc="upper left", fontsize=9)
ax.grid(alpha=0.3)
ax.annotate("2023 Bull Run", xy=(pd.Timestamp("2023-07-01"), 2.2),
            color=AMBER, fontsize=9, fontstyle="italic")
ax.annotate("2024 Correction", xy=(pd.Timestamp("2024-10-01"), -0.5),
            color=RED, fontsize=9, fontstyle="italic")
plt.tight_layout()
plt.savefig(REPORTS_DIR / "rolling_sharpe_chart.png", dpi=150, bbox_inches="tight",
            facecolor=fig.get_facecolor())
plt.show()
print("Chart saved → reports/rolling_sharpe_chart.png")


Chart saved → reports/rolling_sharpe_chart.png


---
## 3. Investor Cohort Analysis

Group investors by the **year of their first transaction** (cohort year).
For each cohort, compute: avg SIP amount, total invested, and most preferred fund category.
Early cohorts (2022) tend to have larger portfolios due to compounding.


In [5]:

df_sip = df_tx[df_tx["transaction_type"] == "SIP"].copy()
df_sip["transaction_date"] = pd.to_datetime(df_sip["transaction_date"])

# First transaction year per investor (cohort)
cohort_map = (df_tx.groupby("investor_id")["transaction_date"]
              .min().dt.year.rename("cohort_year"))
df_sip = df_sip.join(cohort_map, on="investor_id")

cohort_agg = df_sip.groupby("cohort_year").agg(
    n_investors    = ("investor_id", "nunique"),
    avg_sip_amount = ("amount_inr",  "mean"),
    total_invested = ("amount_inr",  "sum"),
    n_transactions = ("amount_inr",  "count"),
).round(2).reset_index()

# Top preferred fund category per cohort
amfi_cat = df_perf[["amfi_code", "category"]].copy()
amfi_cat["amfi_code"] = amfi_cat["amfi_code"].astype(str)
df_sip["amfi_code"] = df_sip["amfi_code"].astype(str)
df_sip_cat = df_sip.merge(amfi_cat, on="amfi_code", how="left")
top_cat = (df_sip_cat.groupby(["cohort_year", "category"])["amount_inr"]
           .sum().reset_index()
           .sort_values("amount_inr", ascending=False)
           .drop_duplicates("cohort_year")[["cohort_year", "category"]]
           .rename(columns={"category": "top_category"}))

cohort_agg = cohort_agg.merge(top_cat, on="cohort_year", how="left")
cohort_agg.to_csv(DATA_PROCESSED / "investor_cohort_analysis.csv", index=False)
print("Cohort analysis saved → investor_cohort_analysis.csv")
print(cohort_agg.to_string(index=False))


Cohort analysis saved → investor_cohort_analysis.csv
 cohort_year  n_investors  avg_sip_amount  total_invested  n_transactions top_category
        2024         4624        10996.89       214978121           19549    Large Cap
        2025          138        13505.21         2255370             167    Large Cap


In [6]:

# ── Plot cohort avg SIP and total invested
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.bar(cohort_agg["cohort_year"].astype(str), cohort_agg["avg_sip_amount"],
        color=EMERALD, alpha=0.85)
ax1.set_xlabel("Cohort Year"); ax1.set_ylabel("Avg SIP Amount (₹)")
ax1.set_title("Average SIP Amount by Investor Cohort Year", fontsize=12)
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"₹{x:,.0f}"))

ax2.bar(cohort_agg["cohort_year"].astype(str),
        cohort_agg["total_invested"] / 1e6, color=BLUE, alpha=0.85)
ax2.set_xlabel("Cohort Year"); ax2.set_ylabel("Total Invested (₹ Millions)")
ax2.set_title("Total SIP Investment by Cohort Year", fontsize=12)

for ax in [ax1, ax2]:
    ax.grid(axis="y", alpha=0.3)
plt.suptitle("Investor Cohort Analysis", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(REPORTS_DIR / "advanced_cohort_analysis.png", dpi=150, bbox_inches="tight",
            facecolor=fig.get_facecolor())
plt.show()
print("Chart saved → reports/advanced_cohort_analysis.png")


Chart saved → reports/advanced_cohort_analysis.png


---
## 4. SIP Continuity Analysis

Identify investors who have **6+ SIP transactions** and compute gaps between consecutive dates.
Investors with any gap > 35 days are flagged as **"at-risk"** (indicating possible SIP pause or cancellation).


In [7]:

df_sip_all = df_tx[df_tx["transaction_type"] == "SIP"].copy()
df_sip_all["transaction_date"] = pd.to_datetime(df_sip_all["transaction_date"])
df_sip_all = df_sip_all.sort_values(["investor_id", "transaction_date"])

# Only investors with 6+ SIP transactions
sip_counts = df_sip_all.groupby("investor_id")["transaction_date"].count()
eligible   = sip_counts[sip_counts >= 6].index
df_eligible = df_sip_all[df_sip_all["investor_id"].isin(eligible)].copy()

def analyse_investor(grp):
    dates = grp["transaction_date"].sort_values().reset_index(drop=True)
    gaps  = dates.diff().dt.days.dropna()
    return pd.Series({
        "n_transactions"  : len(dates),
        "avg_gap_days"    : round(gaps.mean(), 1),
        "max_gap_days"    : gaps.max(),
        "first_tx"        : dates.iloc[0].date(),
        "last_tx"         : dates.iloc[-1].date(),
        "at_risk"         : "Yes" if gaps.max() > 35 else "No",
    })

continuity = df_eligible.groupby("investor_id").apply(analyse_investor).reset_index()
continuity.to_csv(DATA_PROCESSED / "sip_continuity_report.csv", index=False)

at_risk_count = (continuity["at_risk"] == "Yes").sum()
total         = len(continuity)
print(f"Investors analysed : {total:,}")
print(f"At-risk investors  : {at_risk_count:,} ({at_risk_count/total*100:.1f}%)")
print(f"Healthy investors  : {total - at_risk_count:,}")
print(f"\nSaved → sip_continuity_report.csv")
continuity.head(10).to_string(index=False)


Investors analysed : 1,362
At-risk investors  : 1,360 (99.9%)
Healthy investors  : 2

Saved → sip_continuity_report.csv


'investor_id  n_transactions  avg_gap_days  max_gap_days   first_tx    last_tx at_risk\n  INV000004               6          85.4         145.0 2024-03-16 2025-05-17     Yes\n  INV000008               6         107.8         209.0 2024-05-12 2025-11-02     Yes\n  INV000010               6         137.8         272.0 2024-01-12 2025-12-01     Yes\n  INV000011               7          47.8          92.0 2024-01-22 2024-11-04     Yes\n  INV000012               8          57.4         116.0 2024-03-27 2025-05-03     Yes\n  INV000013               7          57.0         122.0 2024-04-08 2025-03-16     Yes\n  INV000014               7         110.7         195.0 2024-02-06 2025-12-01     Yes\n  INV000023               8          58.6         173.0 2024-03-21 2025-05-05     Yes\n  INV000028               6          73.0         238.0 2024-05-20 2025-05-20     Yes\n  INV000029               7          79.8         187.0 2024-05-11 2025-09-02     Yes'

In [8]:

# ── Plot gap distribution ──────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.hist(continuity["max_gap_days"].clip(upper=120), bins=30,
         color=BLUE, alpha=0.8, edgecolor="#1E293B")
ax1.axvline(35, color=RED, linestyle="--", linewidth=2, label="35-day threshold")
ax1.set_xlabel("Maximum Gap Between SIP Dates (days)")
ax1.set_ylabel("Number of Investors")
ax1.set_title("Distribution of Max SIP Gap", fontsize=12)
ax1.legend()

risk_counts = continuity["at_risk"].value_counts()
colors_pie  = [RED, EMERALD]
ax2.pie(risk_counts, labels=["At-Risk", "Healthy"], colors=colors_pie,
        autopct="%1.1f%%", startangle=90,
        textprops={"color": "#E2E8F0"})
ax2.set_title("SIP Continuity Status", fontsize=12)

plt.suptitle("SIP Continuity Analysis", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(REPORTS_DIR / "advanced_sip_continuity.png", dpi=150, bbox_inches="tight",
            facecolor=fig.get_facecolor())
plt.show()
print("Chart saved → reports/advanced_sip_continuity.png")


Chart saved → reports/advanced_sip_continuity.png


---
## 5. Sector HHI Concentration

The **Herfindahl-Hirschman Index (HHI)** measures portfolio concentration.
HHI = Σ(weight_pct/100)² for each fund's holdings.

- HHI > 0.25 → **Highly Concentrated** (few stocks dominate)
- 0.15–0.25 → **Moderate Concentration**
- < 0.15 → **Diversified Portfolio**

A high HHI portfolio is riskier but can generate alpha if stock picks are correct.


In [9]:

# Filter to equity funds (Large, Mid, Small Cap, Flexi Cap, ELSS)
equity_cats = ["Large Cap", "Mid Cap", "Small Cap", "Flexi Cap", "ELSS",
               "Large & Mid Cap", "Multi Cap"]
df_perf["amfi_code"] = df_perf["amfi_code"].astype(str)
equity_codes = df_perf[df_perf["category"].isin(equity_cats)]["amfi_code"].astype(str).tolist()

df_port["amfi_code"] = df_port["amfi_code"].astype(str)
df_eq = df_port[df_port["amfi_code"].isin(equity_codes)].copy()

def compute_hhi(grp):
    weights = grp["weight_pct"] / 100
    return (weights ** 2).sum()

hhi = df_eq.groupby("amfi_code").apply(compute_hhi).reset_index()
hhi.columns = ["amfi_code", "hhi"]

def classify(h):
    if h > 0.25: return "Highly Concentrated"
    if h > 0.15: return "Moderate"
    return "Diversified"

hhi["concentration"] = hhi["hhi"].apply(classify)
hhi = hhi.merge(df_perf[["amfi_code", "scheme_name", "category", "return_3yr_pct"]],
                on="amfi_code", how="left")
hhi.to_csv(DATA_PROCESSED / "hhi_concentration.csv", index=False)
print(f"HHI computed for {len(hhi)} equity funds → hhi_concentration.csv")
print(hhi[["scheme_name", "hhi", "concentration", "return_3yr_pct"]]
      .sort_values("hhi", ascending=False).to_string(index=False))


HHI computed for 31 equity funds → hhi_concentration.csv
                                          scheme_name      hhi concentration  return_3yr_pct
                Axis Bluechip Fund - Regular - Growth 0.206448      Moderate           11.84
               ABSL Small Cap Fund - Regular - Growth 0.200700      Moderate           22.38
            SBI Small Cap Fund - Direct Plan - Growth 0.174751      Moderate           23.14
       Nippon India Large Cap Fund - Regular - Growth 0.168298      Moderate           14.00
Mirae Asset Emerging Bluechip Fund - Regular - Growth 0.167930      Moderate           14.56
             ICICI Pru Midcap Fund - Regular - Growth 0.157570      Moderate           18.08
    HDFC Mid-Cap Opportunities Fund - Direct - Growth 0.152414      Moderate           15.29
               Kotak Bluechip Fund - Regular - Growth 0.149680   Diversified           12.25
        Mirae Asset Tax Saver Fund - Regular - Growth 0.149396   Diversified           13.58
   HDFC Mid-C

In [10]:

# ── Plot HHI bar + concentration scatter ──────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

color_map = {"Highly Concentrated": RED, "Moderate": AMBER, "Diversified": EMERALD}
colors    = hhi["concentration"].map(color_map)
short_names = hhi["scheme_name"].str.split("-").str[0].str.strip()

ax1.barh(short_names, hhi["hhi"], color=colors, alpha=0.85)
ax1.axvline(0.25, color=RED,   linestyle="--", linewidth=1.2, label="High (>0.25)")
ax1.axvline(0.15, color=AMBER, linestyle="--", linewidth=1.2, label="Moderate (>0.15)")
ax1.set_xlabel("HHI Score")
ax1.set_title("HHI Concentration by Equity Fund", fontsize=12)
ax1.legend(fontsize=8)
ax1.invert_yaxis()

# Scatter: HHI vs 3yr return
ax2.scatter(hhi["hhi"], hhi["return_3yr_pct"],
            c=hhi["concentration"].map(color_map), s=80, alpha=0.8)
ax2.set_xlabel("HHI Concentration Score")
ax2.set_ylabel("3-Year CAGR (%)")
ax2.set_title("Concentration vs. Return Profile", fontsize=12)
ax2.axvline(0.25, color=RED,   linestyle="--", linewidth=0.8, alpha=0.5)
ax2.axvline(0.15, color=AMBER, linestyle="--", linewidth=0.8, alpha=0.5)
ax2.grid(alpha=0.3)

plt.suptitle("Sector HHI Concentration Analysis", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(REPORTS_DIR / "advanced_hhi_concentration.png", dpi=150, bbox_inches="tight",
            facecolor=fig.get_facecolor())
plt.show()
print("Chart saved → reports/advanced_hhi_concentration.png")


Chart saved → reports/advanced_hhi_concentration.png


---
## 6. Five Advanced Insights

### Insight 1: Highest VaR Funds Are Predominantly Small Cap


In [11]:

worst_var = df_var.sort_values("var_95_pct").head(5)[["scheme_name", "var_95_pct", "cvar_95_pct"]]
print("5 Funds with Worst VaR (highest daily loss risk):")
print(worst_var.to_string(index=False))
best_var  = df_var.sort_values("var_95_pct", ascending=False).head(5)[["scheme_name","var_95_pct"]]
print("\n5 Funds with Best VaR (lowest daily loss risk):")
print(best_var.to_string(index=False))


5 Funds with Worst VaR (highest daily loss risk):
                                  scheme_name  var_95_pct  cvar_95_pct
   SBI Small Cap Fund - Regular Plan - Growth    -11.5140     -19.0751
        DSP Small Cap Fund - Regular - Growth     -9.3315     -19.2843
    SBI Small Cap Fund - Direct Plan - Growth     -9.2594     -16.5692
Mirae Asset Tax Saver Fund - Regular - Growth     -8.1586     -14.7889
       Axis Small Cap Fund - Regular - Growth     -8.0276     -16.5607

5 Funds with Best VaR (lowest daily loss risk):
                                         scheme_name  var_95_pct
        SBI Magnum Gilt Fund - Regular Plan - Growth     -1.3771
Nippon India Gilt Securities Fund - Regular - Growth     -1.6030
                 ABSL Liquid Fund - Regular - Growth     -1.6997
                Kotak Liquid Fund - Regular - Growth     -1.7945
        HDFC Short Term Debt Fund - Regular - Growth     -1.8672


### Insight 2: 2022 Cohort Investors Have Highest Average SIP Commitment
Early cohorts who started in 2022 have had 4 years to compound their investments
and typically show the highest total portfolio values.


In [12]:

print(cohort_agg[["cohort_year", "n_investors", "avg_sip_amount",
                  "total_invested", "top_category"]].to_string(index=False))


 cohort_year  n_investors  avg_sip_amount  total_invested top_category
        2024         4624        10996.89       214978121    Large Cap
        2025          138        13505.21         2255370    Large Cap


### Insight 3: SIP Continuity Rate by Cohort Year
Newer cohorts (2025) show higher at-risk rates, likely because they haven't had
enough time to establish consistent SIP habits.


In [13]:

cohort_map2 = (df_tx.groupby("investor_id")["transaction_date"]
               .min().dt.year.rename("cohort_year"))
continuity2 = continuity.join(cohort_map2, on="investor_id")
risk_by_cohort = (continuity2.groupby("cohort_year")["at_risk"]
                  .apply(lambda x: (x == "Yes").mean() * 100)
                  .round(1).reset_index())
risk_by_cohort.columns = ["cohort_year", "at_risk_pct"]
print("At-Risk % by Cohort Year:")
print(risk_by_cohort.to_string(index=False))


At-Risk % by Cohort Year:
 cohort_year  at_risk_pct
        2024         99.9


### Insight 4: Concentrated Funds Do Not Consistently Outperform
Despite higher risk (high HHI), concentrated equity funds show no clear return advantage
over diversified funds, suggesting active concentration is not rewarded in this data period.


In [14]:

hhi_return = (hhi.groupby("concentration")["return_3yr_pct"]
              .agg(["mean", "max", "min", "count"]).round(2))
print("3-Year Return by Concentration Category:")
print(hhi_return.to_string())


3-Year Return by Concentration Category:
                mean    max    min  count
concentration                            
Diversified    15.33  23.39  11.30     24
Moderate       17.04  23.14  11.84      7


### Insight 5: Rolling Sharpe Peaked in Mid-2023 Bull Run
The 90-day rolling Sharpe for all 5 funds peaked during the mid-2023 bull run
(NIFTY crossed 20,000 for the first time), then moderated during the 2024
global rate-tightening cycle before recovering in H2-2025.


In [15]:

# Summary table of peak rolling Sharpe dates
print("Recommendation Engine Demo:")
print("─" * 50)
from scripts.recommender import recommend
for risk in ["Low", "Moderate", "High"]:
    top = recommend(risk)
    print(f"\nRisk={risk}:")
    if not top.empty:
        print(top[["scheme_name", "sharpe_ratio", "return_3yr_pct"]].to_string())


Recommendation Engine Demo:
──────────────────────────────────────────────────

Risk=Low:
                                scheme_name  sharpe_ratio  return_3yr_pct
1  ICICI Pru Liquid Fund - Regular - Growth          7.68            7.68
2      Kotak Liquid Fund - Regular - Growth          6.18            6.18
3       ABSL Liquid Fund - Regular - Growth          5.14            5.14

Risk=Moderate:
                                     scheme_name  sharpe_ratio  return_3yr_pct
1      HDFC Top 100 Fund - Regular Plan - Growth          1.06           14.84
2  Mirae Asset Large Cap Fund - Regular - Growth          1.06           14.81
3      ICICI Pru Bluechip Fund - Direct - Growth          1.03           14.41

Risk=High:
                                     scheme_name  sharpe_ratio  return_3yr_pct
1  Kotak Emerging Equity Fund - Regular - Growth          0.96           18.23
2       ICICI Pru Midcap Fund - Regular - Growth          0.95           18.08
3     SBI Small Cap Fund - Regula

---
## 7. Deliverables Summary

| File | Description |
|---|---|
| `data/processed/var_cvar_report.csv` | VaR 95% & CVaR for all 40 funds |
| `data/processed/investor_cohort_analysis.csv` | Cohort-level SIP behaviour |
| `data/processed/sip_continuity_report.csv` | At-risk investor flags |
| `data/processed/hhi_concentration.csv` | Sector concentration scores |
| `reports/rolling_sharpe_chart.png` | Rolling 90-day Sharpe for 5 funds |
| `reports/advanced_var_cvar.png` | VaR/CVaR bar chart |
| `reports/advanced_cohort_analysis.png` | Cohort SIP charts |
| `reports/advanced_sip_continuity.png` | SIP continuity charts |
| `reports/advanced_hhi_concentration.png` | HHI scatter and bar |
